In [51]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
 #   for filename in filenames:
  #      print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image, ImageFile
import os


ImageFile.LOAD_TRUNCATED_IMAGES = True

In [53]:
class FloodDataset(Dataset):
    def __init__(self, image_dir, mask_dir, limit=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.images = sorted(os.listdir(self.image_dir))

        
        if limit is not None:
            self.images = self.images[:limit]

        self.transform = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path)
        mask = Image.open(mask_path)

        image = self.transform(image)
        mask = self.transform(mask)

        
        mask = mask[0]
        mask = (mask > 0).long()

        return image, mask

In [54]:
image_path = "/kaggle/input/datasets/saiharshitjami/flood-images-mask-segmentation/Images"
mask_path  = "/kaggle/input/datasets/saiharshitjami/flood-images-mask-segmentation/Masks"


dataset = FloodDataset(image_path, mask_path, limit=200)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [55]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [56]:
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU()
        )

        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU()
        )

        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU()
        )

        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU()
        )

        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1),
            nn.ReLU()
        )

        self.final = nn.Conv2d(16, 2, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)

In [57]:
model = UNet()

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [58]:
for epoch in range(6):
    model.train()
    total_loss = 0

    for images, masks in train_loader:

        outputs = model(images)

        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss/len(train_loader))

Epoch: 1 Loss: 0.6939521074295044
Epoch: 2 Loss: 0.6919431924819947
Epoch: 3 Loss: 0.6897186875343323
Epoch: 4 Loss: 0.686377876996994
Epoch: 5 Loss: 0.6791961699724197
Epoch: 6 Loss: 0.6698148012161255


In [62]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, masks in test_loader:

        outputs = model(images)

        preds = torch.argmax(outputs, dim=1)

        
        correct += (preds == masks).sum().item()
        total += masks.numel()

accuracy = (correct / total)*100

print("Accuracy:", accuracy)

Accuracy: 70.41671752929688


In [65]:
model.eval()
test_loss = 0

with torch.no_grad():
    for images, masks in test_loader:

        outputs = model(images)

        loss = criterion(outputs, masks)
        test_loss += loss.item()

print("Test Loss:", test_loss/len(test_loader)) # modle eror on unseen data

Test Loss: 0.6577965497970581
